<a href="https://colab.research.google.com/github/ElinaHossain/InstacartBevAnalysis/blob/main/instacart_beverages_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)

try:
    import mlxtend
except ImportError:
    !pip install -q mlxtend

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

from google.colab import files
uploaded = files.upload()

if uploaded:
    print(uploaded.keys())
else:
    print("No files were uploaded.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

KeyboardInterrupt: 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# --- 2.2 Read CSV files ---
# Assumes the files are in the current working directory

orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products__train.csv")
products = pd.read_csv("products.csv")
aisles = pd.read_csv("aisles.csv")
departments = pd.read_csv("departments.csv")

print("orders shape:", orders.shape)
print("order_products shape:", order_products.shape)
print("products shape:", products.shape)
print("aisles shape:", aisles.shape)
print("departments shape:", departments.shape)

orders.head()

## 3. Department selection

Randomly assigning 5 departments based on the student ID, then choosing *one* of them.  
Below is the random selection code followed by the explicit choice of the `beverages` department.


In [ ]:
# --- 3.1 Randomly generate 5 departments based on student ID ---
import random

department = (
    "frozen",
    "bakery",
    "produce",
    "alcohol",
    "international",
    "beverages",
    "pets",
    "dry goods pasta",
    "bulk",
    "personal care",
    "meat seafood",
    "pantry",
    "breakfast",
    "canned goods",
    "dairy eggs",
    "household",
    "babies",
    "snack",
    "deli",
)

student_id_one = 202742533  # <-- Replace with your CSUN student ID
student_id_two = None       # Keep as None for solo project

seed = (student_id_one + student_id_two) // 2 if student_id_two else student_id_one
random.seed(seed)

selected_departments = random.choices(department, k=5)
print("Randomly assigned 5 departments:", selected_departments)

In [ ]:
# --- 3.2 Choose one department from the 5 generated ---
# According to the assignment, we select one of the five printed above.
# In this project, we focus on the 'beverages' department, which is among the assigned ones.

chosen_department_name = "beverages"
print("Chosen department:", chosen_department_name)

## 4. Data preprocessing and merging

We join the product, aisle, and department metadata, then merge with order-level data to create a unified transaction table.


In [ ]:
# --- 4.1 Merge products with aisles and departments ---
prod_full = (
    products
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

prod_full.rename(
    columns={
        "aisle": "aisle_name",
        "department": "department_name"
    },
    inplace=True,
)

prod_full.head()

In [ ]:
# --- 4.2 Merge with order_products and orders (add time info) ---
order_details = (
    order_products
    .merge(prod_full, on="product_id", how="left")
    .merge(orders[["order_id", "order_dow", "order_hour_of_day"]], on="order_id", how="left")
)

order_details.head()

In [ ]:
# --- 4.3 Filter to the chosen department (beverages) ---
dept_df = order_details[order_details["department_name"] == chosen_department_name]
print("Rows for beverages:", dept_df.shape[0])
dept_df.head()

## 5. Exploratory Data Analysis (EDA) for Beverages

We perform several basic statistical analyses and visualizations:

1. Aisles in the Beverages department and number of products per aisle  
2. Top 10 most popular beverage products  
3. Percentage of orders that include at least one beverage item  
4. Temporal patterns: day-of-week and hour-of-day for beverage orders vs all orders  


### 5.1 Aisles in Beverages and number of products

In [ ]:
aisle_counts = (
    prod_full[prod_full["department_name"] == chosen_department_name]
    .groupby("aisle_name")["product_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Number of aisles in beverages:", aisle_counts.shape[0])
aisle_counts

In [ ]:
top_n = 15
ax = aisle_counts.head(top_n).plot(kind="bar", figsize=(10, 4))
ax.set_title("Number of Distinct Products by Aisle – Beverages Department")
ax.set_xlabel("Aisle")
ax.set_ylabel("Number of Products")
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

### 5.2 Top 10 most popular beverage products

In [ ]:
top_products = (
    dept_df
    .groupby("product_name")["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

top_products

In [ ]:
ax = top_products.plot(kind="bar", figsize=(10, 4))
ax.set_title("Top 10 Beverage Products (by Number of Orders)")
ax.set_xlabel("Product")
ax.set_ylabel("Number of Orders")
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

### 5.3 Percentage of orders that include beverages

In [ ]:
orders_with_bev = dept_df["order_id"].nunique()
total_orders = order_details["order_id"].nunique()
pct_bev = orders_with_bev / total_orders * 100

print(f"Orders with at least one beverage item: {orders_with_bev}")
print(f"Total orders: {total_orders}")
print(f"Percentage of orders that include beverages: {pct_bev:.2f}%")

### 5.4 Temporal patterns: day-of-week and hour-of-day

In [ ]:
# Overall distributions
dow_all = orders["order_dow"].value_counts(normalize=True).sort_index()
hour_all = orders["order_hour_of_day"].value_counts(normalize=True).sort_index()

# Orders that include beverages
bev_order_ids = dept_df["order_id"].unique()
orders_with_bev_info = orders[orders["order_id"].isin(bev_order_ids)]

dow_bev = orders_with_bev_info["order_dow"].value_counts(normalize=True).sort_index()
hour_bev = orders_with_bev_info["order_hour_of_day"].value_counts(normalize=True).sort_index()

In [ ]:
# Day-of-week plot
plt.figure(figsize=(8,4))
plt.plot(dow_all.index, dow_all.values, marker="o", label="All orders")
plt.plot(dow_bev.index, dow_bev.values, marker="o", linestyle="--", label="Orders with beverages")
plt.xlabel("Day of Week (0 = Sunday)")
plt.ylabel("Proportion of Orders")
plt.title("Order Distribution by Day of Week")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Hour-of-day plot
plt.figure(figsize=(8,4))
plt.plot(hour_all.index, hour_all.values, marker="o", label="All orders")
plt.plot(hour_bev.index, hour_bev.values, marker="o", linestyle="--", label="Orders with beverages")
plt.xlabel("Hour of Day (0–23)")
plt.ylabel("Proportion of Orders")
plt.title("Order Distribution by Hour of Day")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Association Rule Mining

We now go beyond simple within-department co-purchases and look at:

1. Cross-department rules (department-level)  
2. Aisle-level rules involving beverage aisles (multilevel patterns)  
3. Product-level rules within beverages (including rare but strong patterns)  

We use the **Apriori** algorithm and generate association rules with support, confidence, and lift.


### 6.1 Department-level rules (cross-department)

In [ ]:
# Build department-level baskets: unique departments per order
dept_basket = (
    order_details
    .groupby("order_id")["department_name"]
    .apply(lambda x: list(set(x.dropna())))
)

dept_basket.head()

In [ ]:
# One-hot encode departments using TransactionEncoder
te_dept = TransactionEncoder()
dept_array = te_dept.fit(dept_basket.tolist()).transform(dept_basket.tolist())
dept_ohe = pd.DataFrame(dept_array, columns=te_dept.columns_).astype("uint8")

dept_ohe.shape

In [ ]:
# Run Apriori for departments
dept_itemsets = apriori(
    dept_ohe,
    min_support=0.02,    # departments that appear in at least 2% of orders
    use_colnames=True,
    max_len=3
)

dept_rules = association_rules(
    dept_itemsets,
    metric="confidence",
    min_threshold=0.3
).sort_values("lift", ascending=False)

print("Number of department-level rules:", dept_rules.shape[0])
dept_rules.head()

In [ ]:
# Filter rules where beverages is in antecedents or consequents
def involves_beverages(itemset):
    return "beverages" in itemset

dept_rules_bev = dept_rules[
    dept_rules["antecedents"].apply(involves_beverages)
    | dept_rules["consequents"].apply(involves_beverages)
].sort_values("lift", ascending=False)

print("Department-level rules involving Beverages:", dept_rules_bev.shape[0])
dept_rules_bev.head(10)

### 6.2 Aisle-level rules involving beverage aisles

In [ ]:
# Restrict to orders that contain at least one beverage item
bev_orders = dept_df["order_id"].unique()

aisle_basket_bev = (
    order_details[order_details["order_id"].isin(bev_orders)]
    .groupby("order_id")["aisle_name"]
    .apply(lambda x: list(set(x.dropna())))
)

len(aisle_basket_bev)

In [ ]:
# One-hot encode aisles for those beverage-related orders
te_aisle = TransactionEncoder()
aisle_array = te_aisle.fit(aisle_basket_bev.tolist()).transform(aisle_basket_bev.tolist())
aisle_ohe_bev = pd.DataFrame(aisle_array, columns=te_aisle.columns_).astype("uint8")

print("Aisle matrix shape (beverage orders only):", aisle_ohe_bev.shape)

In [ ]:
# Drop very rare aisles (appear in < 3% of beverage-related orders)
aisle_support = aisle_ohe_bev.mean()
keep_aisles = aisle_support[aisle_support >= 0.03].index

aisle_ohe_small = aisle_ohe_bev[keep_aisles]
print("Reduced aisle matrix shape:", aisle_ohe_small.shape)

In [ ]:
# Run Apriori on reduced aisle matrix (pairs only to control complexity)
aisle_itemsets = apriori(
    aisle_ohe_small,
    min_support=0.03,
    use_colnames=True,
    max_len=2
)

aisle_rules = association_rules(
    aisle_itemsets,
    metric="confidence",
    min_threshold=0.3
).sort_values("lift", ascending=False)

print("Number of aisle-level rules:", aisle_rules.shape[0])
aisle_rules.head()

In [ ]:
# Identify which aisles belong to the beverages department
bev_aisles = set(
    prod_full[prod_full["department_name"] == "beverages"]["aisle_name"].unique()
)
print("Beverage aisles:", bev_aisles)

def involves_bev_aisle(itemset):
    return any(a in bev_aisles for a in itemset)

aisle_rules_bev = aisle_rules[
    aisle_rules["antecedents"].apply(involves_bev_aisle)
    | aisle_rules["consequents"].apply(involves_bev_aisle)
].sort_values("lift", ascending=False)

print("Aisle-level rules involving beverage aisles:", aisle_rules_bev.shape[0])
aisle_rules_bev.head(10)

### 6.3 Product-level rules within beverages

In [ ]:
# Beverage-only subset (order_id, product_name for beverages)
bev_only = dept_df[["order_id", "product_name"]].copy()

# Keep beverage products that appear in at least 50 orders
bev_prod_freq = bev_only.groupby("product_name")["order_id"].nunique()
keep_products = bev_prod_freq[bev_prod_freq >= 50].index

bev_only_frequent = bev_only[bev_only["product_name"].isin(keep_products)]
print("Frequent beverage products:", len(keep_products))

In [ ]:
# Build product baskets for beverage orders
bev_basket = (
    bev_only_frequent
    .groupby("order_id")["product_name"]
    .apply(lambda x: list(set(x)))
)

len(bev_basket), bev_basket.head()

In [ ]:
# One-hot encode beverage products
te_prod = TransactionEncoder()
prod_array = te_prod.fit(bev_basket.tolist()).transform(bev_basket.tolist())
bev_ohe = pd.DataFrame(prod_array, columns=te_prod.columns_).astype("uint8")

print("Beverage product matrix shape:", bev_ohe.shape)

In [ ]:
# Run Apriori on beverage products
bev_itemsets = apriori(
    bev_ohe,
    min_support=0.01,    # products appearing together in at least 1% of beverage orders
    use_colnames=True,
    max_len=2
)

bev_rules = association_rules(
    bev_itemsets,
    metric="confidence",
    min_threshold=0.3
).sort_values("lift", ascending=False)

print("Number of beverage product-level rules:", bev_rules.shape[0])
bev_rules.head()

In [ ]:
# Rare but strong beverage product patterns (low support, high lift)
rare_strong_bev = bev_rules[
    (bev_rules["support"] < 0.02) &
    (bev_rules["lift"] > 2.0)
].sort_values("lift", ascending=False)

print("Rare but strong beverage product rules:", rare_strong_bev.shape[0])
rare_strong_bev.head(10)

In [ ]:
# Negative or substitution-like patterns (lift < 1)
negative_bev = bev_rules[
    (bev_rules["lift"] < 1.0) &
    (bev_rules["confidence"] > 0.2)
].sort_values("lift").head(10)

negative_bev